# Wu-Xing: Complete Comparison of the Bilinear and Saturating Models
### One notebook containing every analysis and every figure

**Contents**

1. The two models, their equilibria and Jacobians
2. Baseline comparison --- eigenvalues in the complex plane
3. Perturbation recovery (the physical stability test)
4. **Stability maps** over parameter space, with boundary contours
5. **Regime maps** --- fixed point / periodic / chaotic
6. Randomised comparison, naive and physiologically ordered
7. Long-run behaviour over random draws
8. Example trajectories
9. Summary

All parameter ranges here are the **physiologically justified** ones
(Section 6), not arbitrary windows.

Runtime -> Run all. About 5-8 minutes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

NAMES=["Wood","Fire","Earth","Metal","Water"]
COLORS=["green","red","magenta","black","blue"]
P_LIN=(1,1,1,1,1,1)          # Lin's own Case 1
P_SAT=(2,0.5,1,0.5,0.3)      # our paper's parameter table


## 1. Models, equilibria, Jacobians

Same graph for both: mother `i-1`, controller `i-2`, child `i+1`, target `i+2`.
They differ only in the functional form of the couplings --- bilinear (mass-action)
versus saturating (demand-limited).


In [ ]:
# ---------------- LIN (bilinear) ----------------
def rhs_lin(t,x,a,b,c,d,g,h):
    x=np.asarray(x)
    return (a*x-b*x**2 + c*x*np.roll(x,1) - d*x*np.roll(x,2)
            - g*x*np.roll(x,-1) - h*x*np.roll(x,-2))

def eq_lin(a,b,c,d,g,h):
    M=b+d+g+h-c
    return a/M if M>0 else np.nan

def jac_lin(xv,a,b,c,d,g,h):
    J=np.zeros((5,5))
    for i in range(5):
        im1,im2,ip1,ip2=(i-1)%5,(i-2)%5,(i+1)%5,(i+2)%5
        J[i,i]=a-2*b*xv[i]+c*xv[im1]-d*xv[im2]-g*xv[ip1]-h*xv[ip2]
        J[i,im1]+=c*xv[i]; J[i,im2]+=-d*xv[i]
        J[i,ip1]+=-g*xv[i]; J[i,ip2]+=-h*xv[i]
    return J

# ---------------- SATURATING (ours) ----------------
def rhs_sat(t,x,a,k,c,g,h):
    x=np.asarray(x)
    return (a*x-k*x**2 + c*np.roll(x,1)/(1+x)
            - g*x/(1+np.roll(x,-2)) - h*x*np.roll(x,2))

def eq_sat(a,k,c,g,h):
    A=k+h; B=(k+h)-a; C=-(a+c-g); D=B*B-4*A*C
    if D<0: return np.nan
    r=(-B+np.sqrt(D))/(2*A)
    return r if r>0 else np.nan

def jac_sat(xv,a,k,c,g,h):
    J=np.zeros((5,5))
    for i in range(5):
        im1,ip2,im2=(i-1)%5,(i+2)%5,(i-2)%5
        mo,ta,co=xv[im1],xv[ip2],xv[im2]
        J[i,i]=a-2*k*xv[i]-c*mo/(1+xv[i])**2-g/(1+ta)-h*co
        J[i,im1]+=c/(1+xv[i]); J[i,ip2]+=g*xv[i]/(1+ta)**2; J[i,im2]+=-h*xv[i]
    return J

def maxre(model,p):
    if model=="lin":
        xe=eq_lin(*p)
        if not np.isfinite(xe) or xe<=0: return np.nan
        return max(np.linalg.eigvals(jac_lin(np.full(5,xe),*p)).real)
    xe=eq_sat(*p)
    if not np.isfinite(xe) or xe<=0: return np.nan
    return max(np.linalg.eigvals(jac_sat(np.full(5,xe),*p)).real)

xl=eq_lin(*P_LIN); xs=eq_sat(*P_SAT)
print(f"Lin  balanced state x* = {xl:.6f}")
print(f"Ours balanced state x* = {xs:.6f}   (closed form 3/4+sqrt(59)/4 = {0.75+np.sqrt(59)/4:.6f})")


## 2. Eigenvalues at the balanced state

Stability requires **every** eigenvalue to have negative real part.


In [ ]:
el=np.linalg.eigvals(jac_lin(np.full(5,xl),*P_LIN))
Js=jac_sat(np.full(5,xs),*P_SAT); es=np.linalg.eigvals(Js)

print("Lin :",", ".join(f"{e.real:+.4f}{e.imag:+.4f}i" for e in sorted(el,key=lambda z:-z.real)))
print(f"      max Re = {max(el.real):+.4f} -> {'UNSTABLE' if max(el.real)>0 else 'stable'}")
print("Ours:",", ".join(f"{e.real:+.4f}{e.imag:+.4f}i" for e in sorted(es,key=lambda z:-z.real)))
print(f"      max Re = {max(es.real):+.4f} -> {'unstable' if max(es.real)>0 else 'STABLE'}")
print("\nOur Jacobian is circulant:",all(np.allclose(Js[i],np.roll(Js[0],i)) for i in range(5)))
w=np.exp(2j*np.pi/5); r0=Js[0]
lam=np.array([sum(r0[m]*w**(j*m) for m in range(5)) for j in range(5)])
print("closed-form DFT eigenvalues match numpy:",np.allclose(np.sort_complex(lam),np.sort_complex(es)))

fig,ax=plt.subplots(figsize=(7,4.6))
ax.axvline(0,color='k',lw=1); ax.axhline(0,color='gray',lw=.5,ls=':')
ax.fill_betweenx([-1.5,1.5],0,3,color='red',alpha=0.07)
ax.text(0.33,1.22,'unstable half-plane',color='red',fontsize=8)
ax.scatter(el.real,el.imag,s=110,facecolor='none',edgecolor='crimson',lw=2,label='Lin (bilinear)')
ax.scatter(es.real,es.imag,s=110,marker='x',color='navy',lw=2,label='Saturating (ours)')
ax.set_xlabel(r"Re $\lambda$"); ax.set_ylabel(r"Im $\lambda$")
ax.set_title("Jacobian eigenvalues at the balanced state")
ax.legend(loc='lower left'); ax.set_xlim(-2.6,0.9); ax.set_ylim(-1.4,1.4); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()


## 3. Perturbation recovery

Eigenvalues are local. Here we disturb one organ and watch what the whole network
does --- the physical meaning of stability.


In [ ]:
def perturb(rhs,params,xstar,mode,t_end=120):
    x0=np.full(5,float(xstar))
    if mode=="small": x0[0]+=0.05*xstar
    else:             x0[0]*=0.30
    s=solve_ivp(rhs,(0,t_end),x0,args=params,method="RK45",
                t_eval=np.linspace(0,t_end,3000),rtol=1e-9,atol=1e-11)
    return s,np.linalg.norm(s.y-xstar,axis=0)

fig,axes=plt.subplots(2,2,figsize=(13,7))
for col,(nm,rhs,p,xst) in enumerate([("Lin (bilinear)",rhs_lin,P_LIN,xl),
                                     ("Saturating (ours)",rhs_sat,P_SAT,xs)]):
    for row,mode in enumerate(["small","severe"]):
        s,dev=perturb(rhs,p,xst,mode); ax=axes[row,col]
        for i in range(5): ax.plot(s.t,s.y[i],color=COLORS[i],label=NAMES[i],lw=1.1)
        ax.axhline(xst,color='gray',ls='--',lw=1)
        ax.set_title(f"{nm} — {mode} perturbation",fontsize=10)
        ax.set_xlabel("time"); ax.set_ylabel("$x_i$")
        if row==0 and col==0: ax.legend(ncol=5,fontsize=6)
plt.tight_layout(); plt.show()

print(f"{'':24s}{'start dev':>11s}{'final dev':>12s}{'outcome':>20s}")
for nm,rhs,p,xst,mode in [("Lin small",rhs_lin,P_LIN,xl,"small"),
                          ("Lin severe",rhs_lin,P_LIN,xl,"severe"),
                          ("Ours small",rhs_sat,P_SAT,xs,"small"),
                          ("Ours severe",rhs_sat,P_SAT,xs,"severe")]:
    _,dev=perturb(rhs,p,xst,mode)
    print(f"{nm:24s}{dev[0]:11.4f}{dev[-1]:12.4f}"
          f"{('recovers' if dev[-1]<1e-3 else 'does NOT recover'):>20s}")


## 4. Stability maps

Colour by $\max_j\operatorname{Re}\lambda_j$ at the balanced state, with the
black **stability boundary** contour at zero. Both equilibrium and Jacobian are
closed-form, so this is pure linear algebra --- fast and exact.

Axes use the physiologically justified ranges: Sheng $c\in[0.3,1.6]$ against Ke
restraint $\in[0.1,1.0]$ (see Section 6 for the justification).


In [ ]:
NG=90
c_vals=np.linspace(0.3,1.6,NG)    # SHENG
r_vals=np.linspace(0.1,1.0,NG)    # RESTR (Lin d, ours h)

Ml=np.full((NG,NG),np.nan); Ms=np.full((NG,NG),np.nan)
for iy,rv in enumerate(r_vals):
    for ix,cv in enumerate(c_vals):
        # Lin: (a,b,c,d,g,h) with SHENG=c, RESTR=d; others at mid-tier
        Ml[iy,ix]=maxre("lin",(1.5,0.65,cv,rv,0.3,0.3))
        # Ours: (a,k,c,g,h) with SHENG=c, RESTR=h; TGT=g at mid-tier
        Ms[iy,ix]=maxre("sat",(1.5,0.65,cv,0.3,rv))

fig,axes=plt.subplots(1,2,figsize=(13,4.8))
# Lin's max Re blows up near M -> 0, which would flatten the colour scale.
# Clip to a robust percentile so the structure is visible in both panels.
vm=np.nanpercentile(np.abs(np.concatenate([Ml.ravel(),Ms.ravel()])),90)
vm=max(vm,0.5)
for ax,Mx,t in [(axes[0],Ml,"Lin (bilinear)"),(axes[1],Ms,"Saturating (ours)")]:
    im=ax.imshow(np.clip(Mx,-vm,vm),origin="lower",aspect="auto",cmap="RdBu_r",vmin=-vm,vmax=vm,
                 extent=[c_vals[0],c_vals[-1],r_vals[0],r_vals[-1]])
    CS=ax.contour(c_vals,r_vals,Mx,levels=[0],colors='k',linewidths=2)
    if len(CS.allsegs[0])>0: ax.clabel(CS,fmt={0:'boundary'},fontsize=8)
    ax.set_xlabel("SHENG  $c$  (nourishment)")
    ax.set_ylabel("RESTR  (Ke restraint received)")
    ax.set_title(f"{t} — stable over {100*np.nanmean(Mx<0):.0f}% of region")
    fig.colorbar(im,ax=ax,label=r"max Re $\lambda$ (clipped)")
plt.suptitle("Stability of the balanced state (blue = stable)")
plt.tight_layout(); plt.show()


## 5. Regime maps

Stability tells you whether balance holds. This tells you what happens **instead**
when it does not: a limit cycle, chaos, or blow-up. Requires integration, so the
grid is coarser.


In [ ]:
def _pk(v): return np.where((v[1:-1]>v[:-2])&(v[1:-1]>v[2:]))[0]+1

# Classify long-run behaviour. Terminates early if the solution diverges past
# `cap` (values that large are non-physiological anyway), which stops blow-up
# cases from grinding the solver to a halt.
def regime(rhs,p,x0=(1,2,3,4,5),T=200,dt=0.1,cap=100.0):
    ev=lambda t,x,*a: np.max(np.abs(x))-cap
    ev.terminal=True
    try:
        s=solve_ivp(rhs,(0,T),list(x0),args=p,method="LSODA",
                    t_eval=np.arange(0,T,dt),rtol=1e-6,atol=1e-8,events=ev)
    except Exception: return "blowup"
    if s.status==1 or not s.success: return "blowup"
    tail=s.y[:,int(s.y.shape[1]*0.6):]
    if tail.size==0 or (not np.all(np.isfinite(tail))) or np.max(np.abs(tail))>cap:
        return "blowup"
    if (tail.max(1)-tail.min(1)).max()<1e-3: return "fixed"
    v=tail[1]; pk=_pk(v)
    if len(pk)<4: return "periodic"
    iv=np.diff(pk)*dt; ht=v[pk]
    if np.std(iv)/(np.mean(iv)+1e-9)<0.06 and np.std(ht)/(abs(np.mean(ht))+1e-9)<0.06:
        return "periodic"
    return "chaotic"

NR=7    # raise for a finer regime map (slower)
cr=np.linspace(0.3,1.6,NR); rr=np.linspace(0.1,1.0,NR)
CODE={"fixed":0,"periodic":1,"chaotic":2,"blowup":3}
CMAP=plt.matplotlib.colors.ListedColormap(["#4c72b0","#55a868","#c44e52","#cccccc"])

Gl=np.zeros((NR,NR)); Gs=np.zeros((NR,NR))
for iy,rv in enumerate(rr):
    for ix,cv in enumerate(cr):
        Gl[iy,ix]=CODE[regime(rhs_lin,(1.5,0.65,cv,rv,0.3,0.3))]
        Gs[iy,ix]=CODE[regime(rhs_sat,(1.5,0.65,cv,0.3,rv))]

fig,axes=plt.subplots(1,2,figsize=(13,4.8))
for ax,G,t in [(axes[0],Gl,"Lin (bilinear)"),(axes[1],Gs,"Saturating (ours)")]:
    im=ax.imshow(G,origin="lower",aspect="auto",cmap=CMAP,vmin=0,vmax=3,
                 extent=[cr[0],cr[-1],rr[0],rr[-1]])
    ax.set_xlabel("SHENG  $c$"); ax.set_ylabel("RESTR"); ax.set_title(t)
cb=fig.colorbar(im,ax=axes,ticks=[0.4,1.1,1.9,2.6])
cb.ax.set_yticklabels(["fixed","periodic","chaotic","blow-up"])
plt.suptitle("Regime maps: long-run behaviour")
plt.show()


## 6. Randomised comparison, with a justified parameter ordering

A single parameter setting proves little, so we draw many at random. But drawing
every coefficient from the same range is not physiologically neutral --- it treats
self-production and pathological over-acting as equally likely to dominate.

**Concept mapping** (the two models name the same quantities differently):

| Concept | Meaning | Lin | Ours |
|---|---|---|---|
| `SELF`  | self-production | $a$ | $a$ |
| `SAT`   | saturation | $b$ | $k$ |
| `SHENG` | nourishment from mother | $c$ | $c$ |
| `RESTR` | Ke restraint received | $d$ | $h$ |
| `TGT`   | over-acting cost toward target | $h$ | $g$ |
| `CHILD` | cost of nourishing child | $g$ | --- |

**Ordering:** $\texttt{SELF}>\texttt{SHENG}>\texttt{RESTR}>\texttt{TGT}$, because

1. an organ's own vital qi is the primary source of its function; relations modulate it;
2. Sheng sustains while Ke merely regulates --- a brake exceeding the engine is a shutdown;
3. normal relations outrank pathological ones (over-acting 相乘 is a deviation);
4. and in the full model 相乘 (贼邪, "greatly weakens") outranks 相侮 (微邪, "mildly
   weakens") --- recorded here, applied when the Wu term is present.

**Fairness:** Lin's balanced state only exists when $b+d+g+h>c$; draws where it does
not are discarded so he is scored only where his own model is well posed.


In [ ]:
TIERS={"SELF":(1.0,2.0),"SHENG":(0.5,1.5),"SAT":(0.3,1.0),
       "RESTR":(0.2,0.8),"TGT":(0.1,0.5),"CHILD":(0.1,0.5)}

def to_params(model,d):
    if model=="lin": return (d["SELF"],d["SAT"],d["SHENG"],d["RESTR"],d["CHILD"],d["TGT"])
    return (d["SELF"],d["SAT"],d["SHENG"],d["TGT"],d["RESTR"])

def draw(r,scheme,lo=0.1,hi=2.0):
    if scheme=="tiered":
        return {k:r.uniform(*v) for k,v in TIERS.items()}
    while True:
        S,A,SH,RE,TG,CH=r.uniform(lo,hi,6)
        if scheme=="uniform": pass
        elif scheme=="full_order":
            if not (S>SH>RE>TG): continue
        return dict(SELF=S,SAT=A,SHENG=SH,RESTR=RE,TGT=TG,CHILD=CH)

def mc(scheme,N=4000,seed=3):
    r=np.random.default_rng(seed)
    out={m:{"stable":0,"valid":0,"vals":[]} for m in ("lin","sat")}
    for _ in range(N):
        d=draw(r,scheme)
        for m in ("lin","sat"):
            v=maxre(m,to_params(m,d))
            if np.isfinite(v):
                out[m]["valid"]+=1; out[m]["stable"]+=(v<0); out[m]["vals"].append(v)
    return out

schemes=[("uniform","uniform (naive)"),("full_order","full ordering"),("tiered","tiered ranges")]
R={}
print(f"{'scheme':>18s}{'Lin':>9s}{'Ours':>9s}")
for key,lab in schemes:
    R[key]=mc(key,2500)   # raise N for tighter error bars
    l=100*R[key]['lin']['stable']/max(R[key]['lin']['valid'],1)
    s=100*R[key]['sat']['stable']/max(R[key]['sat']['valid'],1)
    print(f"{lab:>18s}{l:>8.1f}%{s:>8.1f}%")

fig,ax=plt.subplots(1,2,figsize=(13,4.4))
x=np.arange(len(schemes)); w=0.38
L=[100*R[k]['lin']['stable']/max(R[k]['lin']['valid'],1) for k,_ in schemes]
S=[100*R[k]['sat']['stable']/max(R[k]['sat']['valid'],1) for k,_ in schemes]
b1=ax[0].bar(x-w/2,L,w,label="Lin",color="#c44e52")
b2=ax[0].bar(x+w/2,S,w,label="Ours",color="#4c72b0")
for b in list(b1)+list(b2):
    ax[0].text(b.get_x()+b.get_width()/2,b.get_height()+1.5,f"{b.get_height():.0f}%",
               ha="center",fontsize=8,fontweight="bold")
ax[0].set_xticks(x); ax[0].set_xticklabels([l for _,l in schemes],fontsize=9)
ax[0].set_ylabel("% of draws with a stable balanced state"); ax[0].set_ylim(0,112)
ax[0].set_title("Stability under three sampling priors"); ax[0].legend()

for k,lab,col in [("tiered","Ours","#4c72b0")]:
    pass
for m,lab,col in [("lin","Lin","#c44e52"),("sat","Ours","#4c72b0")]:
    v=np.array(R["tiered"][m]["vals"]); v=np.clip(v,-4,3)
    ax[1].hist(v,bins=70,alpha=0.6,label=lab,color=col,density=True)
ax[1].axvline(0,color='k',lw=2)
ax[1].set_xlabel(r"max Re $\lambda$"); ax[1].set_ylabel("density")
ax[1].set_title("Leading eigenvalue (tiered ranges)"); ax[1].legend()
plt.tight_layout(); plt.show()

print("\nNote: the ordering RAISES Lin's score — the physiological prior is")
print("favourable to the competing model, making this the conservative comparison.")


## 7. Long-run behaviour over random draws

What happens when the balanced state is *not* stable?


In [ ]:
M=200
r=np.random.default_rng(1)
keys=["fixed","periodic","chaotic","blowup"]
cl={k:0 for k in keys}; cs={k:0 for k in keys}
for _ in range(M):
    d=draw(r,"tiered")
    cl[regime(rhs_lin,to_params("lin",d))]+=1
    cs[regime(rhs_sat,to_params("sat",d))]+=1

print(f"{'regime':>10s}{'Lin':>8s}{'Ours':>8s}")
for k in keys: print(f"{k:>10s}{100*cl[k]/M:>7.0f}%{100*cs[k]/M:>7.0f}%")

fig,ax=plt.subplots(figsize=(8.5,3.8))
cols={"fixed":"#4c72b0","periodic":"#55a868","chaotic":"#c44e52","blowup":"#999999"}
bl=np.zeros(2)
for k in keys:
    vals=np.array([100*cl[k]/M,100*cs[k]/M])
    ax.barh(["Lin (bilinear)","Saturating (ours)"],vals,left=bl,color=cols[k],label=k)
    for i,(v,b) in enumerate(zip(vals,bl)):
        if v>4: ax.text(b+v/2,i,f"{v:.0f}%",ha="center",va="center",
                        color="white",fontweight="bold",fontsize=9)
    bl+=vals
ax.set_xlabel("% of random draws (tiered ranges)"); ax.set_xlim(0,100)
ax.set_title(f"Long-run behaviour over {M} physiologically sampled draws")
ax.legend(ncol=4,loc="lower center",bbox_to_anchor=(0.5,-0.42))
plt.tight_layout(); plt.show()


## 8. Example trajectories

In [ ]:
def show(rhs,p,xstar,title):
    x0=np.full(5,float(xstar)) if np.isfinite(xstar) else np.array([1.,2,3,4,5])
    x0[0]*=1.05
    s=solve_ivp(rhs,(0,150),x0,args=p,method="RK45",
                t_eval=np.linspace(0,150,4000),rtol=1e-9,atol=1e-11)
    fig,ax=plt.subplots(1,2,figsize=(12,3.3))
    for i in range(5): ax[0].plot(s.t,s.y[i],color=COLORS[i],label=NAMES[i],lw=1)
    if np.isfinite(xstar): ax[0].axhline(xstar,color='gray',ls='--',lw=1)
    ax[0].set_xlabel("time"); ax[0].set_ylabel("$x_i$"); ax[0].set_title(title)
    ax[0].legend(ncol=5,fontsize=6)
    n0=len(s.t)//2
    ax[1].plot(s.y[1,n0:],s.y[2,n0:],lw=0.6,color="purple")
    ax[1].set_xlabel("Fire"); ax[1].set_ylabel("Earth"); ax[1].set_title("phase (settled)")
    plt.tight_layout(); plt.show()

show(rhs_lin,P_LIN,xl,"LIN at its baseline — perturbation grows (unstable)")
show(rhs_sat,P_SAT,xs,"OURS at baseline — returns to balance (stable)")


## 9. Summary

| | Lin (bilinear) | Saturating (ours) |
|---|---|---|
| balanced state | $1/3$ | $\tfrac34+\tfrac{\sqrt{59}}4\approx2.6703$ |
| $\max\operatorname{Re}\lambda$ at baseline | $+0.206$ (unstable) | $-1.154$ (stable) |
| small perturbation | grows $\sim50\times$ | decays to zero |
| severe perturbation | organs collapse | full recovery |
| stable, naive prior | $\sim23\%$ | $\sim95\%$ |
| stable, ordered prior | $\sim45\%$ | $\sim92\%$ |
| stable, tiered ranges | $\sim24\%$ | $\sim99\%$ |
| long-run chaotic | $\sim25\%$ | $\sim0\%$ |

Health --- a state the body returns to after disturbance --- is representable in the
saturating model and essentially not in the bilinear one. The difference traces to a
single modelling choice: replacing unbounded mass-action couplings with saturating,
demand-limited ones. The conclusion holds under every sampling prior tested,
including the one most favourable to the competing model.
